# Exercise 5 — CNNs vs. Vision Transformers

**Course:** CAS Deep Learning — Computer Vision  
**Day:** 2 | **Block:** 8

## Learning objectives
After this exercise you should be able to:
- Compare CNN and ViT architectures on a practical task
- Understand how attention maps differ from CNN activation maps
- Reason about data efficiency and compute trade-offs
- Make an informed backbone selection for a given scenario

## Instructions
- Run cells top to bottom.
- Fill in code where you see `# YOUR CODE HERE`.
- Estimated time: ~30–40 minutes.

In [ ]:
import sys

import pyrootutils

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q -r requirements.txt
    from google.colab import drive

    ROOT_PATH = "/content/drive"
    drive.mount(ROOT_PATH)
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")
DATA_PATH.mkdir(parents=True, exist_ok=True)

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import timm
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder

## Load a Sample Image

We load a single camera trap image to use for all visualizations in this exercise. Unlike previous exercises where we trained models, here we focus on **understanding what the models see** — so one image is enough.

In [ ]:
# Default: Serengeti balanced dataset (same as Exercises 1, 2, and 4)
DATASET_PATH = DATA_PATH / "ser" / "ser_balanced"

# --- Alternative: CCT-20 dataset (8 balanced classes) ---
# Uncomment the block below to use CCT-20 instead of Serengeti.
#
# def load_camera_trap(data_path, source="cct20"):
#     """Download and extract a curated camera trap dataset subset."""
#     from pathlib import Path
#     repos = {
#         "cct20": ("marco-willi/camera-trap-cct20", "cct20.tar.gz"),
#         "kgalagadi": ("marco-willi/camera-trap-kgalagadi", "kgalagadi.tar.gz"),
#     }
#     repo_id, filename = repos[source]
#     dataset_dir = Path(data_path) / source
#     if dataset_dir.exists():
#         print(f"Dataset already present: {dataset_dir}")
#         return dataset_dir
#     import tarfile
#     from huggingface_hub import hf_hub_download
#     archive = hf_hub_download(repo_id, filename, repo_type="dataset")
#     with tarfile.open(archive) as tar:
#         tar.extractall(data_path)
#     return dataset_dir
#
# DATASET_PATH = load_camera_trap(DATA_PATH, source="cct20")

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# Standard ImageNet preprocessing — both models will use 256x256 input
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

eval_transforms = transforms.Compose(
    [
        transforms.Resize(292),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

# Load the val split and pick a sample image that contains an animal
val_dataset = ImageFolder(root=DATASET_PATH / "val", transform=eval_transforms)
raw_dataset = ImageFolder(root=DATASET_PATH / "val")  # without transforms, for display
class_names = val_dataset.classes

# Find the first non-empty image
SAMPLE_IDX = 0
for i, (_, label) in enumerate(val_dataset.imgs):
    if class_names[label] != "empty":
        SAMPLE_IDX = i
        break

sample_tensor, sample_label = val_dataset[SAMPLE_IDX]
sample_pil, _ = raw_dataset[SAMPLE_IDX]

# Pre-crop the PIL image to match the tensor (for overlays later)
center_crop = transforms.Compose([transforms.Resize(292), transforms.CenterCrop(256)])
sample_cropped = center_crop(sample_pil)

print(f"Sample image: class = {class_names[sample_label]}")
print(f"Tensor shape: {sample_tensor.shape}")

In [ ]:
# Display the sample image
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
_ = ax.imshow(sample_cropped)
_ = ax.set_title(f"Sample image: {class_names[sample_label]}", fontsize=12)
_ = ax.axis("off")
plt.show()

## Part 1 — Load and Compare Architectures

We compare two vision backbone families:

- **ResNet-50** — a CNN built from stacked convolutional layers. Processes images through local receptive fields that gradually grow. The classic workhorse.
- **ViT-Little/16** — a compact Vision Transformer that splits the image into 16x16 patches and processes them with self-attention. Uses 4 **register tokens** (extra learnable tokens that help the model organize internal information) and **global average pooling** instead of a CLS token.

Both are pretrained on ImageNet and available via `timm`.

**Before you start:** Which model do you think has more parameters — a "standard" CNN or a "little" Transformer?

Load both models from `timm` in feature-extraction mode (`num_classes=0`) and print for each:
- Model class name
- Total number of parameters
- Input resolution

Store the models in `cnn_model` and `vit_model`.

<details>
<summary>Hint</summary>

```python
MODEL_ID = "vit_little_patch16_reg4_gap_256.sbb_in1k"
cnn_model = timm.create_model("resnet50", pretrained=True, num_classes=0)
vit_model = timm.create_model(MODEL_ID, pretrained=True, num_classes=0)
```

</details>

In [ ]:
# YOUR CODE HERE
MODEL_ID = "vit_little_patch16_reg4_gap_256.sbb_in1k"

cnn_model = timm.create_model("resnet50", pretrained=True, num_classes=0)
cnn_model = cnn_model.eval().to(device)

vit_model = timm.create_model(MODEL_ID, pretrained=True, num_classes=0)
vit_model = vit_model.eval().to(device)

cnn_params = sum(p.numel() for p in cnn_model.parameters())
vit_params = sum(p.numel() for p in vit_model.parameters())

print(f"{'Model':<30} {'Class':<20} {'Parameters':>15}")
print("-" * 67)
print(f"{'ResNet-50 (CNN)':<30} {cnn_model.__class__.__name__:<20} {cnn_params:>15,}")
print(
    f"{'ViT-Little/16 (Transformer)':<30} {vit_model.__class__.__name__:<20} {vit_params:>15,}"
)
print(
    f"\nViT has {vit_params / cnn_params:.1f}x {'more' if vit_params > cnn_params else 'fewer'} parameters than ResNet-50"
)

In [ ]:
assert cnn_params > 0 and vit_params > 0, "Models should have parameters"
print(f"CNN: {cnn_params:,} params | ViT: {vit_params:,} params")
print("OK")

**Question:** Which model has more parameters? Is this what you expected?

<details>
<summary>Click to reveal answer</summary>

**Surprise — the ViT-Little has roughly the same number of parameters as ResNet-50** (~22M each). The "little" variant is deliberately compact. Yet despite its small size, it achieves competitive accuracy thanks to the efficiency of self-attention and modern training recipes (DeiT-style distillation, register tokens).

The key insight: **parameter count alone doesn't determine a model's power.** Architecture matters — ViTs and CNNs use their parameters very differently:
- CNNs spend parameters on many stacked convolutional filters with local receptive fields
- ViTs spend parameters on attention matrices that relate *every* patch to *every* other patch globally

This means a ViT can capture long-range relationships with fewer parameters than a CNN would need (which has to build them up through depth).

</details>

## Part 2 — Inference Speed

Parameters tell us about model *capacity*, but what about *speed*? Let's measure how long each model takes to process a batch of images on CPU.

**Before you start:** Both models have similar parameter counts — do you think they'll be equally fast?

Time the forward pass for each model on a batch of 16 random 256x256 images **on CPU**. Run each model **3 times** and report the **average** time per image in milliseconds.

Store results in `cnn_ms_per_image` and `vit_ms_per_image`.

<details>
<summary>Hint</summary>

```python
bench_device = "cpu"
batch = torch.randn(16, 3, 256, 256, device=bench_device)
model_cpu = model.to(bench_device)
times = []
for _ in range(3):
    t0 = time.time()
    with torch.no_grad():
        _ = model_cpu(batch)
    times.append(time.time() - t0)
ms_per_image = np.mean(times) / 16 * 1000
```

</details>

In [ ]:
# YOUR CODE HERE
BATCH_SIZE = 16
NUM_RUNS = 3

# Force CPU for fair comparison — GPU parallelism hides architectural differences
bench_device = "cpu"
cnn_bench = cnn_model.to(bench_device)
vit_bench = vit_model.to(bench_device)
batch = torch.randn(BATCH_SIZE, 3, 256, 256, device=bench_device)


def benchmark(model, batch, num_runs=3):
    """Benchmark forward pass. Returns average ms per image."""
    # Warm-up run
    with torch.no_grad():
        _ = model(batch)

    times = []
    for _ in range(num_runs):
        t0 = time.time()
        with torch.no_grad():
            _ = model(batch)
        times.append(time.time() - t0)

    avg_time = np.mean(times)
    return avg_time / batch.shape[0] * 1000  # ms per image


cnn_ms_per_image = benchmark(cnn_bench, batch, NUM_RUNS)
vit_ms_per_image = benchmark(vit_bench, batch, NUM_RUNS)

# Move models back to the main device
cnn_model = cnn_model.to(device)
vit_model = vit_model.to(device)

print(f"ResNet-50:    {cnn_ms_per_image:.1f} ms/image")
print(f"ViT-Little:   {vit_ms_per_image:.1f} ms/image")
print(
    f"\nViT is {vit_ms_per_image / cnn_ms_per_image:.1f}x {'slower' if vit_ms_per_image > cnn_ms_per_image else 'faster'} than ResNet-50 on CPU"
)

In [ ]:
assert cnn_ms_per_image > 0, "Timing should be positive"
assert vit_ms_per_image > 0, "Timing should be positive"
print(f"CNN: {cnn_ms_per_image:.1f} ms | ViT: {vit_ms_per_image:.1f} ms")
print("OK")

In [ ]:
# Visualize the comparison
models = ["ResNet-50\n(CNN)", "ViT-Little/16\n(Transformer)"]
params = [cnn_params / 1e6, vit_params / 1e6]
speeds = [cnn_ms_per_image, vit_ms_per_image]
colors = ["#4C72B0", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Parameters
bars = axes[0].bar(models, params, color=colors)
for bar, p in zip(bars, params, strict=True):
    _ = axes[0].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.3,
        f"{p:.0f}M",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[0].set_ylabel("Parameters (millions)")
_ = axes[0].set_title("Model Size")
_ = axes[0].grid(True, alpha=0.3, axis="y")

# Speed
bars = axes[1].bar(models, speeds, color=colors)
for bar, s in zip(bars, speeds, strict=True):
    _ = axes[1].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.5,
        f"{s:.1f} ms",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[1].set_ylabel("Time per image (ms)")
_ = axes[1].set_title("Inference Speed (CPU)")
_ = axes[1].grid(True, alpha=0.3, axis="y")

_ = fig.suptitle("ResNet-50 vs. ViT-Little: Size and Speed", fontsize=14, y=1.02)
_ = plt.tight_layout()
plt.show()

**Question:** How do the inference speeds compare? What architectural properties explain the difference?

<details>
<summary>Click to reveal answer</summary>

The result depends on hardware and model size. On this machine, the compact ViT-Little is competitive with — or even faster than — ResNet-50 despite using self-attention. This is because:

1. **ViT-Little is genuinely small** — 14 layers with 320-dim embeddings and only 5 heads. The self-attention matrices are modest: 260x260 per head per layer.

2. **Self-attention parallelizes well.** On hardware that can exploit this (GPUs, but also modern CPUs with wide SIMD), attention can be efficient.

3. **ResNet-50 has 50 layers** of convolutions with large intermediate feature maps (especially in the early stages), which creates memory pressure.

The story changes with larger ViTs: **ViT-Base** (86M params, 12 heads, 768-dim) is noticeably slower than ResNet-50 because attention scales quadratically with token count. The key insight is that **architecture matters more than parameter count** for inference speed.

</details>

## Part 3 — Attention Maps (ViT)

A Vision Transformer doesn't process images with convolutions — it uses **self-attention**. Each patch can attend to every other patch, and the attention weights tell us which patches the model considers relevant.

We can extract these attention weights and visualize them as a heatmap. This reveals **what the model looks at** when it processes an image.

**Before you start:** What do you think the ViT attends to — the animal, the background, or the whole image equally?

### How attention extraction works

Our ViT-Little model has some modern features that affect attention extraction:

- **No CLS token** — uses **global average pooling** (GAP) over all patch tokens instead
- **4 register tokens** — extra learnable tokens that help the model organize attention patterns
- **5 attention heads** per layer (ViT-Base has 12)

In `timm`'s ViT, each transformer block has an `attn` module with a `qkv` projection. We register a forward hook that recomputes the attention weights from Q and K:

```
model.blocks[-1].attn
        │
        ├── qkv projection → Q, K, V
        │
        └── attention = softmax(Q @ K^T / √d)
                shape: (batch, heads, tokens, tokens)
```

For our model on a 256x256 image:
- 16x16 = 256 image patches + 4 register tokens = **260 tokens**
- **5 attention heads** per layer
- Attention shape: `(1, 5, 260, 260)`

In [ ]:
# Prepare the sample image as a batch
input_tensor = sample_tensor.unsqueeze(0).to(device)  # (1, 3, 256, 256)
print(f"Input shape: {input_tensor.shape}")

Register a forward hook on the **last** attention block that captures the attention weights. Since our model doesn't expose attention weights directly, we recompute them from the QKV projection inside the hook.

Then extract a spatial attention map:
1. Take the attention from `attn_weights[:, :, prefix:, prefix:]` — only patch-to-patch attention, shape `(5, 256, 256)` (5 heads, 256 query patches, 256 key patches)
2. Average across heads → shape `(256, 256)` — one row per query patch
3. Average across query patches → shape `(256,)` — shows which patches receive the most attention overall
4. Reshape to `(16, 16)` — the spatial grid of patches

Store the result in `attn_map` (a 16x16 NumPy array).

<details>
<summary>Click to reveal solution</summary>

```python
captured = {}

def make_hook(storage):
    def hook_fn(module, input, output):
        B, N, C = input[0].shape
        num_heads = module.num_heads
        head_dim = C // num_heads
        qkv = module.qkv(input[0]).reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, _ = qkv.unbind(0)
        if hasattr(module, "q_norm") and module.q_norm is not None:
            q = module.q_norm(q)
        if hasattr(module, "k_norm") and module.k_norm is not None:
            k = module.k_norm(k)
        attn = (q @ k.transpose(-2, -1)) * (head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        storage["attn"] = attn.detach().cpu()
    return hook_fn

hook = vit_model.blocks[-1].attn.register_forward_hook(make_hook(captured))

with torch.no_grad():
    _ = vit_model(input_tensor)

hook.remove()

attn_weights = captured["attn"]  # (1, 5, 260, 260)

# Extract patch-to-patch attention (skip register tokens)
num_prefix = vit_model.num_prefix_tokens  # 4 registers
patch_attn = attn_weights[0, :, num_prefix:, num_prefix:]  # (5, 256, 256)
patch_attn_avg = patch_attn.mean(dim=0)  # (256, 256)
spatial_attn = patch_attn_avg.mean(dim=0)  # (256,) — which patches receive most attention
attn_map = spatial_attn.reshape(16, 16).numpy()
```

</details>

In [ ]:
# YOUR CODE HERE
captured = {}


def make_hook(storage):
    """Create a hook that recomputes attention weights from QKV."""

    def hook_fn(module, input, output):
        B, N, C = input[0].shape
        num_heads = module.num_heads
        head_dim = C // num_heads
        qkv = (
            module.qkv(input[0])
            .reshape(B, N, 3, num_heads, head_dim)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, _ = qkv.unbind(0)
        # Apply QK normalization if present
        if hasattr(module, "q_norm") and module.q_norm is not None:
            q = module.q_norm(q)
        if hasattr(module, "k_norm") and module.k_norm is not None:
            k = module.k_norm(k)
        attn = (q @ k.transpose(-2, -1)) * (head_dim**-0.5)
        attn = attn.softmax(dim=-1)  # (B, heads, N, N)
        storage["attn"] = attn.detach().cpu()

    return hook_fn


hook = vit_model.blocks[-1].attn.register_forward_hook(make_hook(captured))

with torch.no_grad():
    _ = vit_model(input_tensor)

hook.remove()

attn_weights = captured["attn"]  # (1, 5, 260, 260)
print(f"Attention weights shape: {attn_weights.shape}")

# Extract patch-to-patch attention (skip register prefix tokens)
num_prefix = vit_model.num_prefix_tokens  # 4 registers
n_patches = 16  # 256/16 patch grid

patch_attn = attn_weights[0, :, num_prefix:, num_prefix:]  # (5, 256, 256)
patch_attn_avg = patch_attn.mean(dim=0)  # (256, 256) — averaged across heads
spatial_attn = patch_attn_avg.mean(
    dim=0
)  # (256,) — which patches receive most attention
attn_map = spatial_attn.reshape(n_patches, n_patches).numpy()

print(f"Prefix tokens (registers): {num_prefix}")
print(f"Patch grid: {n_patches}x{n_patches} = {n_patches**2} patches")
print(f"Attention map: {attn_map.shape}")

In [ ]:
assert attn_weights.shape == (1, 5, 260, 260), (
    f"Expected (1, 5, 260, 260), got {attn_weights.shape}"
)
assert attn_map.shape == (16, 16), f"Expected (16, 16), got {attn_map.shape}"
assert attn_map.min() >= 0, "Attention weights should be non-negative (post-softmax)"
print("OK")

In [ ]:
# Overlay the attention map on the original image
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
_ = axes[0].imshow(sample_cropped)
_ = axes[0].set_title("Original Image", fontsize=12)
_ = axes[0].axis("off")

# Raw attention map
im = axes[1].imshow(attn_map, cmap="viridis")
_ = axes[1].set_title("ViT Attention (16x16)", fontsize=12)
_ = axes[1].axis("off")
_ = plt.colorbar(im, ax=axes[1], fraction=0.046)

# Attention overlaid on image — upscale to 256x256
attn_upscaled = np.array(
    PIL.Image.fromarray(attn_map).resize((256, 256), resample=PIL.Image.BILINEAR)
)

_ = axes[2].imshow(sample_cropped)
_ = axes[2].imshow(attn_upscaled, cmap="jet", alpha=0.5)
_ = axes[2].set_title("Attention Overlay", fontsize=12)
_ = axes[2].axis("off")

_ = plt.suptitle("ViT Attention Map — Where Does the Model Look?", fontsize=14)
_ = plt.tight_layout()
plt.show()

**Question:** Which parts of the image does the ViT attend to? Is this what you expected?

<details>
<summary>Click to reveal answer</summary>

The ViT typically attends most to the **main subject** of the image — in a camera trap photo, this is usually the animal. The attention map highlights semantically important regions while ignoring background clutter (grass, trees, sky).

This is remarkable because the model was pretrained on ImageNet, not camera trap images. It has learned a general concept of "foreground object" that transfers across domains. The patch tokens with highest received attention tend to be the ones carrying the most distinctive visual information.

Note: unlike a CLS-based ViT where we look at "what the CLS token attends to", this model uses global average pooling. So we visualize which patches receive the most attention from all other patches — the "most attended-to" regions.

</details>

Let's also look at what individual attention heads focus on. Different heads often specialize in different aspects of the image.

In [ ]:
# Show attention maps for all 5 heads
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for head_idx, ax in enumerate(axes.flat):
    # Per-head: average across query patches
    head_attn = patch_attn[head_idx].mean(dim=0)  # (256,)
    head_map = head_attn.reshape(16, 16).numpy()
    _ = ax.imshow(sample_cropped)
    _ = ax.imshow(
        np.array(
            PIL.Image.fromarray(head_map).resize(
                (256, 256), resample=PIL.Image.BILINEAR
            )
        ),
        cmap="jet",
        alpha=0.5,
    )
    _ = ax.set_title(f"Head {head_idx}", fontsize=10)
    _ = ax.axis("off")

_ = plt.suptitle(
    "Individual Attention Heads — Each Head Sees Something Different", fontsize=14
)
_ = plt.tight_layout()
plt.show()

**Question:** Do all heads attend to the same region, or do they specialize?

<details>
<summary>Click to reveal answer</summary>

**Different heads specialize in different things.** Some heads focus on the foreground object (the animal), others attend to background structure, edges, or textures. A few heads may distribute attention broadly across the image ("global" heads), while others concentrate on very specific regions.

This specialization is why multi-head attention is powerful — each head captures a different "view" of the relationships between patches. The model combines all these views to build a rich representation. This is similar to how different convolutional filters detect different features, but attention heads operate globally rather than locally.

</details>

## Part 4 — Activation Maps (CNN)

CNNs don't have attention — they process images through stacked convolutional layers with local receptive fields. But we can still visualize **what the CNN responds to** by looking at the activation maps (feature maps) from the last convolutional block.

Each channel in the feature map detects a different pattern. By averaging across all channels, we get a single heatmap showing which spatial regions activate the CNN most strongly.

Register a forward hook on `cnn_model.layer4` (the last residual block of ResNet-50). Run the forward pass on `input_tensor` and capture the output feature maps.

Then:
1. Average across channels → shape `(8, 8)`
2. Convert to a NumPy array

Store the result in `act_map` (an 8x8 NumPy array).

<details>
<summary>Hint: the hook pattern is simpler than Part 3 — just capture the output directly</summary>

```python
captured_cnn = {}

def cnn_hook_fn(module, input, output):
    captured_cnn["features"] = output.detach()

hook = cnn_model.layer4.register_forward_hook(cnn_hook_fn)
```

</details>

In [ ]:
# YOUR CODE HERE
captured_cnn = {}


def cnn_hook_fn(module, input, output):
    """Capture the feature maps from the last residual block."""
    captured_cnn["features"] = output.detach()


hook = cnn_model.layer4.register_forward_hook(cnn_hook_fn)

with torch.no_grad():
    _ = cnn_model(input_tensor)

hook.remove()

feature_maps = captured_cnn["features"]  # (1, 2048, 8, 8)
print(f"Feature maps shape: {feature_maps.shape}")

# Average across channels to get a spatial activation heatmap
act_map = feature_maps[0].mean(dim=0).cpu().numpy()  # (8, 8)
print(f"Activation map: {act_map.shape}")

In [ ]:
assert feature_maps.shape == (1, 2048, 8, 8), (
    f"Expected (1, 2048, 8, 8), got {feature_maps.shape}"
)
assert act_map.shape == (8, 8), f"Expected (8, 8), got {act_map.shape}"
print("OK")

In [ ]:
# Overlay the activation map on the original image
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
_ = axes[0].imshow(sample_cropped)
_ = axes[0].set_title("Original Image", fontsize=12)
_ = axes[0].axis("off")

# Raw activation map
im = axes[1].imshow(act_map, cmap="viridis")
_ = axes[1].set_title("CNN Activations (8x8)", fontsize=12)
_ = axes[1].axis("off")
_ = plt.colorbar(im, ax=axes[1], fraction=0.046)

# Activation overlaid on image
act_upscaled = np.array(
    PIL.Image.fromarray(act_map).resize((256, 256), resample=PIL.Image.BILINEAR)
)
_ = axes[2].imshow(sample_cropped)
_ = axes[2].imshow(act_upscaled, cmap="jet", alpha=0.5)
_ = axes[2].set_title("Activation Overlay", fontsize=12)
_ = axes[2].axis("off")

_ = plt.suptitle("CNN Activation Map — Where Does ResNet-50 Respond?", fontsize=14)
_ = plt.tight_layout()
plt.show()

## Part 5 — CNN vs. ViT: Side by Side

Now the key comparison: how do the ViT attention map and the CNN activation map differ when looking at the same image?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original
_ = axes[0].imshow(sample_cropped)
_ = axes[0].set_title(f"Original ({class_names[sample_label]})", fontsize=12)
_ = axes[0].axis("off")

# ViT attention
_ = axes[1].imshow(sample_cropped)
_ = axes[1].imshow(attn_upscaled, cmap="jet", alpha=0.5)
_ = axes[1].set_title("ViT — Attention Map", fontsize=12)
_ = axes[1].axis("off")

# CNN activation
_ = axes[2].imshow(sample_cropped)
_ = axes[2].imshow(act_upscaled, cmap="jet", alpha=0.5)
_ = axes[2].set_title("CNN — Activation Map", fontsize=12)
_ = axes[2].axis("off")

_ = plt.suptitle(
    "What Does Each Model Look At?",
    fontsize=14,
)
_ = plt.tight_layout()
plt.show()

**Question:** What are the structural differences between the ViT attention map and the CNN activation map?

<details>
<summary>Click to reveal answer</summary>

Several key differences:

1. **Resolution:** The CNN activation map is 8x8 (coarse) while the ViT attention map is 16x16 (higher). Both are low-resolution compared to the input (256x256), but ViT's patch-based tokenization preserves more spatial detail.

2. **Scope:** ViT attention is **global** — each patch can attend to any other patch, so the attention map can highlight distant, disconnected regions. CNN activations are **local** — they build up from small receptive fields, so the highlighted regions tend to be spatially contiguous.

3. **Interpretation:** ViT attention shows which patches *receive the most attention* from all other patches. CNN activation shows which spatial locations *fire strongly* after many layers of filtering. These are fundamentally different mechanisms for identifying important image regions.

4. **Sharpness:** ViT attention maps often look more "discrete" (patch-by-patch), while CNN activation maps tend to be smoother due to the overlapping receptive fields.

</details>

Let's also see how both models respond to more images.

In [ ]:
def get_vit_attention(model, image_tensor, device="cpu"):
    """Extract the patch-averaged attention map from the last ViT block."""
    captured = {}

    def make_hook(storage):
        def hook_fn(module, input, output):
            B, N, C = input[0].shape
            num_heads = module.num_heads
            head_dim = C // num_heads
            qkv = (
                module.qkv(input[0])
                .reshape(B, N, 3, num_heads, head_dim)
                .permute(2, 0, 3, 1, 4)
            )
            q, k, _ = qkv.unbind(0)
            if hasattr(module, "q_norm") and module.q_norm is not None:
                q = module.q_norm(q)
            if hasattr(module, "k_norm") and module.k_norm is not None:
                k = module.k_norm(k)
            attn = (q @ k.transpose(-2, -1)) * (head_dim**-0.5)
            attn = attn.softmax(dim=-1)
            storage["attn"] = attn.detach().cpu()

        return hook_fn

    hook = model.blocks[-1].attn.register_forward_hook(make_hook(captured))
    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0).to(device))
    hook.remove()

    # Extract patch-to-patch attention
    num_prefix = model.num_prefix_tokens
    n_patches_side = int((captured["attn"].shape[-1] - num_prefix) ** 0.5)
    patch_attn = captured["attn"][0, :, num_prefix:, num_prefix:]  # (heads, P, P)
    spatial = patch_attn.mean(dim=0).mean(dim=0)  # (P,)
    return spatial.reshape(n_patches_side, n_patches_side).numpy()


def get_cnn_activation(model, image_tensor, device="cpu"):
    """Extract the channel-averaged activation map from the last ResNet block."""
    captured = {}

    def hook_fn(module, input, output):
        captured["features"] = output.detach()

    hook = model.layer4.register_forward_hook(hook_fn)
    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0).to(device))
    hook.remove()

    return captured["features"][0].mean(dim=0).cpu().numpy()


def upscale(heatmap, size=256):
    """Upscale a small heatmap to the target size using bilinear interpolation."""
    return np.array(
        PIL.Image.fromarray(heatmap).resize((size, size), resample=PIL.Image.BILINEAR)
    )

In [ ]:
# Pick 4 images from different classes
sample_indices = []
seen_labels = set()
for i, (_, label) in enumerate(val_dataset.imgs):
    if label not in seen_labels and class_names[label] != "empty":
        sample_indices.append(i)
        seen_labels.add(label)
    if len(sample_indices) == 4:
        break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for col, idx in enumerate(sample_indices):
    img_tensor, label = val_dataset[idx]
    img_pil, _ = raw_dataset[idx]
    img_cropped = center_crop(img_pil)

    vit_attn = get_vit_attention(vit_model, img_tensor, device)
    cnn_act = get_cnn_activation(cnn_model, img_tensor, device)

    # Row 0: Original
    _ = axes[0, col].imshow(img_cropped)
    _ = axes[0, col].set_title(class_names[label], fontsize=11, fontweight="bold")
    _ = axes[0, col].axis("off")

    # Row 1: ViT attention
    _ = axes[1, col].imshow(img_cropped)
    _ = axes[1, col].imshow(upscale(vit_attn), cmap="jet", alpha=0.5)
    _ = axes[1, col].axis("off")

    # Row 2: CNN activation
    _ = axes[2, col].imshow(img_cropped)
    _ = axes[2, col].imshow(upscale(cnn_act), cmap="jet", alpha=0.5)
    _ = axes[2, col].axis("off")

_ = axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
_ = axes[1, 0].set_ylabel("ViT Attention", fontsize=12, fontweight="bold")
_ = axes[2, 0].set_ylabel("CNN Activation", fontsize=12, fontweight="bold")

_ = plt.suptitle("ViT vs. CNN — What Each Model Looks At", fontsize=14)
_ = plt.tight_layout()
plt.show()

## Part 6 — Scenario Reasoning

Now that you've compared CNNs and ViTs hands-on, answer the following questions. Think about your answer before revealing the solution.

**Question 1:** You have **500 labeled training images** and no pretrained model. Which backbone do you choose — CNN or ViT?

<details>
<summary>Click to reveal answer</summary>

**CNN (e.g., ResNet-50).** With only 500 images and no pretraining, the CNN's inductive biases (local connectivity, translation equivariance) act as strong regularizers. A ViT trained from scratch on 500 images would severely overfit because it has fewer built-in assumptions about image structure and must learn spatial relationships from data.

</details>

**Question 2:** You have **500,000 labeled training images** and plenty of GPU budget. Which backbone?

<details>
<summary>Click to reveal answer</summary>

**ViT (or a hybrid).** With enough data, ViTs can learn the spatial relationships that CNNs get for free from their architecture — and potentially learn *better* ones. ViTs have consistently outperformed CNNs on large-scale benchmarks when data is abundant. The flexibility of self-attention allows the model to capture long-range dependencies that CNNs struggle with.

</details>

**Question 3:** You need to run inference **on a mobile phone** with limited RAM and compute. Which backbone?

<details>
<summary>Click to reveal answer</summary>

**A lightweight CNN** such as MobileNet or EfficientNet-Lite. These architectures are specifically designed for mobile deployment with techniques like depthwise separable convolutions and neural architecture search. Standard ViTs are slower than CNNs of comparable size on CPU-bound devices — as we saw in Part 2, even the compact ViT-Little is slower than ResNet-50 on CPU despite having similar parameter counts.

That said, efficient ViT variants (MobileViT, EfficientViT) are closing the gap.

</details>

**Question 4:** You care most about **retrieval quality** (finding visually similar images). Would you prefer a supervised or self-supervised pretrained model?

<details>
<summary>Click to reveal answer</summary>

**Self-supervised** (e.g., DINOv2). As we saw in Exercise 3, self-supervised models learn representations that capture fine-grained visual similarity — shape, texture, colour, style — rather than just category boundaries. Supervised models are optimized for classification, which means two very different-looking chairs might get similar embeddings simply because they are both "chairs."

For retrieval, you want the embedding space to reflect *visual similarity*, not just *semantic categories*. This is exactly what self-supervised objectives optimize for.

</details>

## Summary

In this exercise you compared CNN and ViT backbones head-to-head:

1. **Architecture comparison** — ViT-Little and ResNet-50 have similar parameter counts (~22M), but very different designs
2. **Inference speed** — CNNs are faster on CPU thanks to local convolutions; ViTs pay for global attention even at small model sizes
3. **Attention maps** — ViTs explicitly attend to image regions; we extracted and visualized patch-to-patch attention patterns
4. **Activation maps** — CNNs produce spatial feature maps; averaging across channels reveals which regions activate strongly
5. **Scenario reasoning** — the right backbone depends on dataset size, compute budget, and task

**Key takeaways:**
- CNNs have built-in spatial inductive biases → more data-efficient, faster inference
- ViTs learn spatial relationships from data → more flexible, better at scale
- Modern compact ViTs (like ViT-Little with registers) can match CNN parameter counts while keeping the benefits of self-attention
- Visualization (attention/activation maps) is essential for understanding *what* a model has learned